In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle

In [2]:
df = pd.read_csv('datasets/Churn_Modelling.csv')

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('EstimatedSalary', axis=1), df['EstimatedSalary'], test_size=0.25, random_state=42)

In [6]:
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [7]:
le = LabelEncoder()
X_train['Gender'] = le.fit_transform(X_train['Gender'])
X_test['Gender'] = le.transform(X_test['Gender'])

In [8]:
ohe = OneHotEncoder()
ohe_tr = ohe.fit_transform(X_train[['Geography']]).toarray()
ohedf_tr = pd.DataFrame(ohe_tr, columns=ohe.get_feature_names_out(), index=X_train.index)

In [9]:
ohe_te = ohe.transform(X_test[['Geography']]).toarray()
ohedf_te = pd.DataFrame(ohe_te, columns=ohe.get_feature_names_out(), index=X_test.index)

In [10]:
X_train = pd.concat([ohedf_tr, X_train.drop('Geography', axis=1)], axis=1)
X_test = pd.concat([ohedf_te, X_test.drop('Geography', axis=1)], axis=1)

In [11]:
X_train.head()

,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
4901,1.0,0.0,0.0,673,1,59,0,178058.06,2,0,1,1
4375,0.0,1.0,0.0,850,1,41,8,60880.68,1,1,0,0
6698,1.0,0.0,0.0,725,0,31,6,0.00,1,0,0,0
9805,1.0,0.0,0.0,644,1,33,7,174571.36,1,0,1,0
1101,0.0,0.0,1.0,703,1,29,9,0.00,2,1,0,0


In [12]:
X_test.head()

,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
6252,0.0,1.0,0.0,596,1,32,3,96709.07,2,0,0,0
4684,1.0,0.0,0.0,623,1,43,1,0.00,2,1,1,0
1731,0.0,0.0,1.0,601,0,44,4,0.00,2,1,0,0
4742,0.0,1.0,0.0,506,1,59,8,119152.10,2,1,1,0
4521,0.0,0.0,1.0,560,0,27,7,124995.98,1,1,1,0


In [13]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

### ANN Implementation

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [15]:
X_train.shape[1],

(12,)

In [16]:
model = Sequential(
    [
        Dense(64, activation='relu', input_shape = (X_train.shape[1],)), ## Initial+ Layer 1
        Dense(32, activation='relu'),  ## layer 2
        Dense(1)  ## Output Layer
    ]
)

c:\Users\Ekangsh\OneDrive\Desktop\Projects\Customer Churn ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.MeanAbsoluteError()

In [19]:
model.compile(optimizer=opt, loss=loss, metrics=['mae'])

In [20]:
## Logging
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import datetime

In [21]:
log_dir = 'log/fitreg/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [22]:
## Early Stopper
early_stopper_callback = EarlyStopping(monitor='val_loss', patience= 20, restore_best_weights=True)

In [23]:
history = model.fit(
    X_train, y_train, 
    validation_data = (X_test, y_test), 
    epochs= 100,
    callbacks = [early_stopper_callback, tensorflow_callback]

)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 83751.2031 - mae: 83751.2031 - val_loss: 51875.6172 - val_mae: 51875.6172
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 50505.9531 - mae: 50505.9531 - val_loss: 50380.6719 - val_mae: 50380.6719
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 49997.8398 - mae: 49997.8398 - val_loss: 50131.9727 - val_mae: 50131.9727
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49829.0586 - mae: 49829.0586 - val_loss: 50175.1328 - val_mae: 50175.1328
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 49714.1641 - mae: 49714.1641 - val_loss: 50103.9414 - val_mae: 50103.9414
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49638.4180 - mae: 49638.4180 - val_loss: 50146.6797 - val_mae: 50146.6797
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49609.5352 - mae: 49609.5352 - val_loss: 50030.7188 - val_mae: 50030.7188
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - 

In [24]:
model.save('regressionmodel.h5')

In [30]:
%reload_ext tensorboard


In [44]:
%reload_ext tensorboard
%tensorboard --logdir log/fitreg

Reusing TensorBoard on port 6006 (pid 30236), started 0:00:36 ago. (Use '!kill 30236' to kill it.)

In [47]:
!taskkill /PID 30236 /F

SUCCESS: The process with PID 30236 has been terminated.
